# SciPRISM-X Methodology

## 1. Environment and Unified Entry Point

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
assert PROJECT_ROOT is not None, "pyproject.toml was not found in the current directory or its parents"
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from sciprism_x.config import load_run_config
from sciprism_x.formal.config import load_formal_config, resolve_scale_contract
from sciprism_x.formal.results import REQUIRED_FORMAL_ARTIFACTS
from sciprism_x.formal.workflow import FORMAL_STAGE_ORDER, run_formal_experiment

## 2. Pinned Environment and Execution Scale

In [ ]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

SCALE = os.environ.get("SCIPRISM_SCALE", "paper")
assert SCALE in {"smoke", "paper"}
default_output = Path("D:/SciPRISM-X-runs") / f"formal-{SCALE}"
OUTPUT_ROOT = Path(os.environ.get("SCIPRISM_OUTPUT", str(default_output))).resolve()
RESUME = True

environment = {
    "scale": SCALE,
    "output_root": str(OUTPUT_ROOT),
    "python": sys.version,
    "platform": platform.platform(),
    "uv_lock_sha256": sha256_file(PROJECT_ROOT / "uv.lock"),
    "requirements_lock_sha256": sha256_file(PROJECT_ROOT / "requirements.lock.txt"),
    "numpy": importlib.metadata.version("numpy"),
    "pandas": importlib.metadata.version("pandas"),
    "torch": importlib.metadata.version("torch"),
}
display(pd.Series(environment, name="resolved_environment"))

## 3. Method Parameter Specification

In [ ]:
config = load_run_config(PROJECT_ROOT / "configs" / "track_b.yaml")
formal_config = load_formal_config(PROJECT_ROOT / "configs" / "formal_execution.yaml")
contract = resolve_scale_contract(config, formal_config, scale=SCALE)

parameter_table = pd.DataFrame(
    {
        "path": item.path,
        "value": item.value,
        "provenance": item.provenance.value,
    }
    for item in config.parameter_records()
)
display(parameter_table.groupby("provenance").size().rename("parameter_count"))
display(parameter_table)
display(pd.Series({name: getattr(contract, name) for name in contract.__dataclass_fields__}, name="resolved_scale_contract"))

## 4. Run the Complete Formal Workflow

In [ ]:
receipt = run_formal_experiment(
    config,
    formal_config,
    scale=SCALE,
    output_root=OUTPUT_ROOT,
    resume=RESUME,
)
RUN_DIR = receipt.final_path
display(pd.Series({
    "run_id": receipt.run_id,
    "scale": receipt.scale,
    "final_path": str(RUN_DIR),
    "computed_response_count": receipt.computed_response_count,
    "applicable_metric_cells": receipt.applicable_metric_cells,
    "nonfinite_metric_cells": receipt.nonfinite_metric_cells,
    "paper_sources_in_computed_rows": receipt.paper_sources_in_computed_rows,
}, name="formal_run_receipt"))

## 5. Integrity and Provenance Checks

In [ ]:
assert receipt.completed_stages == FORMAL_STAGE_ORDER
assert receipt.nonfinite_metric_cells == 0
assert receipt.paper_sources_in_computed_rows == 0
assert set(dict(receipt.artifact_hashes)) == set(REQUIRED_FORMAL_ARTIFACTS)

stage_table = pd.DataFrame(
    {
        "stage": stage.name,
        "resumed": stage.resumed,
        "content_hash": stage.content_hash,
    }
    for stage in receipt.stage_receipts
)
display(stage_table)

## 6. Load All Computed Results

In [ ]:
computed = RUN_DIR / "computed"
responses = pd.read_parquet(computed / "responses.parquet")
tuning_responses = pd.read_parquet(computed / "tuning_responses.parquet")
per_case_metrics = pd.read_parquet(computed / "per_case_metrics.parquet")
tuning_metrics = pd.read_parquet(computed / "tuning_metrics.parquet")
baseline_selection = json.loads(
    (computed / "baseline_selection.json").read_text(encoding="utf-8")
)
baselines = pd.read_parquet(computed / "baselines.parquet")
baseline_boundaries = pd.read_parquet(computed / "baseline_boundaries.parquet")
baseline_open_set = pd.read_parquet(computed / "baseline_open_set.parquet")
ablations = pd.read_parquet(computed / "ablations.parquet")
boundaries = pd.read_parquet(computed / "boundaries.parquet")
open_set = pd.read_parquet(computed / "open_set.parquet")
bootstrap_results = pd.read_parquet(computed / "bootstrap_results.parquet")
significance_results = pd.read_parquet(computed / "significance_results.parquet")
aggregate_metrics = json.loads(
    (computed / "aggregate_metrics.json").read_text(encoding="utf-8")
)
implementation_contract = json.loads(
    (computed / "implementation_contract.json").read_text(encoding="utf-8")
)

display(pd.Series({
    "responses": len(responses),
    "tuning_responses": len(tuning_responses),
    "per_case_metrics": len(per_case_metrics),
    "tuning_metrics": len(tuning_metrics),
    "baseline_tuned_methods": len(baseline_selection["trials"]),
    "baselines": len(baselines),
    "baseline_boundaries": len(baseline_boundaries),
    "baseline_open_set": len(baseline_open_set),
    "ablations": len(ablations),
    "boundaries": len(boundaries),
    "open_set_rows": len(open_set),
    "bootstrap_rows": len(bootstrap_results),
    "significance_rows": len(significance_results),
}, name="computed_table_rows"))
display(pd.Series(aggregate_metrics, name="computed_aggregate"))

## 7. Numeric Finiteness and Paired-Result Checks

In [ ]:
required_metrics = [
    "tsr", "pcs", "residual_consistency",
    "node_loss", "structure_loss", "constraint_loss",
]
assert np.isfinite(per_case_metrics[required_metrics].to_numpy(dtype=float)).all()
assert set(per_case_metrics["source"]) == {"computed"}
assert set(tuning_metrics["split"]) == {"train", "validation"}
assert set(tuning_metrics["original_task_id"]).isdisjoint(set(per_case_metrics["original_task_id"]))
assert set(ablations["source"]) == {"computed"}
assert np.isfinite(ablations[["main_value", "metric_value", "delta"]].to_numpy(dtype=float)).all()

display(per_case_metrics[required_metrics + ["source"]].describe())
display(ablations.groupby(["ablation", "metric"])["delta"].agg(["count", "mean", "median"]))
display(significance_results)

## 8. Reference and Computed Values

In [ ]:
published_metrics = json.loads(
    (RUN_DIR / "paper_reference" / "published_metrics.json").read_text(encoding="utf-8")
)
discrepancy_report = pd.read_csv(RUN_DIR / "comparison" / "discrepancy_report.csv")
display(discrepancy_report)
display(pd.Series({"published_reference_entries": len(published_metrics)}))

## 9. Save Results and Hash Manifest

In [ ]:
saved_receipt = json.loads(
    (RUN_DIR / "formal_run_receipt.json").read_text(encoding="utf-8")
)
artifact_manifest = pd.DataFrame(
    sorted(saved_receipt["artifact_hashes"].items()),
    columns=["artifact", "sha256"],
)
display(artifact_manifest)

## 10. Deterministic Plotting Entry Point

In [ ]:
def load_plot_inputs(run_dir: Path) -> dict[str, object]:
    computed_dir = run_dir / "computed"
    return {
        "responses": pd.read_parquet(computed_dir / "responses.parquet"),
        "tuning_responses": pd.read_parquet(computed_dir / "tuning_responses.parquet"),
        "per_case_metrics": pd.read_parquet(computed_dir / "per_case_metrics.parquet"),
        "tuning_metrics": pd.read_parquet(computed_dir / "tuning_metrics.parquet"),
        "baseline_selection": json.loads(
            (computed_dir / "baseline_selection.json").read_text(encoding="utf-8")
        ),
        "baselines": pd.read_parquet(computed_dir / "baselines.parquet"),
        "baseline_boundaries": pd.read_parquet(computed_dir / "baseline_boundaries.parquet"),
        "baseline_open_set": pd.read_parquet(computed_dir / "baseline_open_set.parquet"),
        "ablations": pd.read_parquet(computed_dir / "ablations.parquet"),
        "boundaries": pd.read_parquet(computed_dir / "boundaries.parquet"),
        "open_set": pd.read_parquet(computed_dir / "open_set.parquet"),
        "bootstrap_results": pd.read_parquet(computed_dir / "bootstrap_results.parquet"),
        "significance_results": pd.read_parquet(computed_dir / "significance_results.parquet"),
        "sample_manifest": pd.read_parquet(computed_dir / "sample_manifest.parquet"),
        "per_instance_outputs": pd.read_parquet(computed_dir / "per_instance_outputs.parquet"),
        "state_assignments": pd.read_parquet(computed_dir / "state_assignments.parquet"),
        "effective_hyperparameters": json.loads(
            (computed_dir / "effective_hyperparameters.json").read_text(encoding="utf-8")
        ),
        "learned_modules": json.loads(
            (computed_dir / "learned_modules.json").read_text(encoding="utf-8")
        ),
        "solver_environment": json.loads(
            (computed_dir / "solver_environment.json").read_text(encoding="utf-8")
        ),
        "protocol_manifest": json.loads(
            (computed_dir / "protocol_manifest.json").read_text(encoding="utf-8")
        ),
        "task_graphs_path": computed_dir / "task_graphs.jsonl",
        "response_graphs_path": computed_dir / "response_graphs.jsonl",
        "validator_outputs_path": computed_dir / "validator_outputs.parquet",
        "transport_plans_path": computed_dir / "transport_plans.npz",
        "transport_diagnostics_path": computed_dir / "transport_diagnostics.parquet",
        "residual_components_path": computed_dir / "residual_components.parquet",
        "aggregate_metrics": json.loads(
            (computed_dir / "aggregate_metrics.json").read_text(encoding="utf-8")
        ),
        "implementation_contract": json.loads(
            (computed_dir / "implementation_contract.json").read_text(encoding="utf-8")
        ),
        "discrepancy_report": pd.read_csv(
            run_dir / "comparison" / "discrepancy_report.csv"
        ),
        "figure_manifest": json.loads(
            (run_dir / "figures" / "figure_manifest.json").read_text(encoding="utf-8")
        ),
        "aggregate_metrics_svg": run_dir / "figures" / "aggregate_metrics.svg",
        "ablation_delta_heatmap_svg": run_dir / "figures" / "ablation_delta_heatmap.svg",
    }

plot_inputs = load_plot_inputs(RUN_DIR)
display(pd.Series({
    name: len(value) if hasattr(value, "__len__") else 1
    for name, value in plot_inputs.items()
}, name="plot_input_size"))